# SurgPhase — Cholec80 Data Exploration

This notebook explores the Cholec80 dataset statistics, visualises surgical phase distributions, and demonstrates the data loading pipeline.

**Prerequisites:**
- Cholec80 dataset downloaded to `/data/cholec80`
- `pip install -e ".[notebooks]"` installed

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='husl')

from src.data.cholec_dataset import Cholec80Dataset, CHOLEC80_PHASES, CHOLEC80_INSTRUMENTS

DATA_ROOT = '/data/cholec80'  # Update this path

## 1. Phase Distribution

In [ ]:
# Load train dataset
train_ds = Cholec80Dataset(
    root=DATA_ROOT,
    split='train',
    temporal_window=None,  # Load full videos
    stride=1,
)

# Collect all phase labels
all_phases = []
for vid_id, df in train_ds.annotations.items():
    all_phases.extend(df['phase_idx'].tolist())

phase_counts = pd.Series(all_phases).value_counts().sort_index()
phase_names_short = [p.replace('Gallbladder', 'GB') for p in CHOLEC80_PHASES]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(len(CHOLEC80_PHASES)), phase_counts.values / 1e6, color=plt.cm.Set2.colors[:7])
ax.set_xticks(range(len(CHOLEC80_PHASES)))
ax.set_xticklabels(phase_names_short, rotation=20, ha='right')
ax.set_ylabel('Frames (millions)')
ax.set_title('Cholec80 Training Set — Phase Distribution (Videos 1–40)')

for bar, count in zip(bars, phase_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{count/1e6:.1f}M', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../assets/phase_distribution.png', bbox_inches='tight')
plt.show()

print(f'Total training frames: {sum(phase_counts.values):,}')
print(f'Class imbalance ratio (max/min): {phase_counts.max() / phase_counts.min():.1f}x')

## 2. Video Duration Statistics

In [ ]:
# Video lengths in minutes (at 1 fps sampling)
video_lengths = {vid: len(df) / 60 for vid, df in train_ds.annotations.items()}

lengths = list(video_lengths.values())
print(f'Training video duration statistics:')
print(f'  Mean:   {np.mean(lengths):.1f} min')
print(f'  Median: {np.median(lengths):.1f} min')
print(f'  Min:    {np.min(lengths):.1f} min')
print(f'  Max:    {np.max(lengths):.1f} min')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(lengths, bins=15, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(np.mean(lengths), color='crimson', linestyle='--', label=f'Mean: {np.mean(lengths):.0f} min')
ax.set_xlabel('Video Duration (minutes)')
ax.set_ylabel('Count')
ax.set_title('Cholec80 — Distribution of Video Durations (Train split)')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Phase Timeline for an Example Video

In [ ]:
PHASE_COLORS = plt.cm.Set2.colors[:7]

vid_id = 1
df = train_ds.annotations[vid_id]
phases = df['phase_idx'].values
time_min = np.arange(len(phases)) / 60  # Assuming 1fps

fig, ax = plt.subplots(figsize=(14, 3))
for i in range(1, len(phases)):
    p = phases[i]
    if p >= 0:
        ax.axvspan(time_min[i-1], time_min[i], alpha=0.85, color=PHASE_COLORS[p])

patches = [mpatches.Patch(color=PHASE_COLORS[i], label=CHOLEC80_PHASES[i]) for i in range(7)]
ax.legend(handles=patches, loc='upper right', ncol=2, fontsize=8)
ax.set_xlabel('Time (minutes)')
ax.set_yticks([])
ax.set_title(f'Video {vid_id:02d} — Phase Timeline')
plt.tight_layout()
plt.show()

## 4. Instrument Co-occurrence Matrix

In [ ]:
# Compute instrument co-occurrence (which instruments appear together)
cooccurrence = np.zeros((7, 7))

for vid_id, df in train_ds.annotations.items():
    tool_cols = [c for c in CHOLEC80_INSTRUMENTS if c in df.columns]
    if len(tool_cols) == 0:
        continue
    tool_matrix = df[tool_cols].values.astype(int)
    cooccurrence += tool_matrix.T @ tool_matrix

# Normalise by diagonal (joint probability)
cooccurrence_norm = cooccurrence / (cooccurrence.diagonal()[:, None] + 1e-6)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cooccurrence_norm, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(7))
ax.set_yticks(range(7))
ax.set_xticklabels(CHOLEC80_INSTRUMENTS, rotation=40, ha='right')
ax.set_yticklabels(CHOLEC80_INSTRUMENTS)
plt.colorbar(im, ax=ax, label='P(col | row)')
ax.set_title('Instrument Co-occurrence Matrix (Cholec80 Training Set)')
plt.tight_layout()
plt.show()